# Glacier evolution using OGGM + whatif

In [ ]:
from pathlib import Path
import xarray as xr 
import pandas as pd
import yaml 
import os

import plotly.graph_objects as go
from   plotly.subplots import make_subplots

wd = Path('/home/rooda/OneDrive/Projects/Whatif-Andes/')
os.chdir(wd) 

# import config and local module
with open("config.yml") as f:
    cfg = yaml.safe_load(f)

## Data

In [ ]:
warming_levels = ["1.5ºC", "2.8ºC", "4.0ºC"]
ensemble_sizes = [7, 5, 6]  # N for each warming level

# volume regional data 
hist_data   = xr.open_dataset("data/glacier/glacier_evolution_2000_2024.nc").volume
future_data = xr.open_dataset("data/glacier/glacier_evolution_2025_2099.nc").volume
future_data["time"] = future_data["time"] + 1 # TODO check
hist_ref  = hist_data.sel(time=2020)
hist_data = hist_data / hist_ref
future_data = future_data / hist_ref

# Volume observations (GLAMBIE)
glambie_data = pd.read_csv("data/glacier/GLAMBIE_southern_andes.csv").set_index('end_dates')
glambie_data['volume_delta_km3'] = glambie_data['combined_gt'] / 0.85
glambie_data['volume_error_km3'] = glambie_data['combined_gt_errors'] / 0.85
glambie_data['volume_total'] = 5340 + glambie_data['volume_delta_km3'].cumsum()
glambie_data['volume_norm'] = glambie_data['volume_total'] / glambie_data.loc[2020, 'volume_total']


# glacier runoff regional data (+ what-if)
glacier_hist_data = xr.open_dataset("data/glacier/run_output_historical.nc").sum(dim='gauge_id').glacier_melt
files = list(Path("data/glacier/").glob("run_output_historical_*.nc"))
whatif_data       = xr.open_mfdataset(files, concat_dim = "warming_level",  combine='nested' )
whatif_data       = whatif_data.sum(dim='gauge_id').glacier_melt.load()
whatif_data       = whatif_data.assign_coords(warming_level=[f.name.split('_')[3] for f in files])

# Apply 15-day moving average
glacier_hist_data = glacier_hist_data.rolling(date=15, center=True).mean()
glacier_hist_data = glacier_hist_data.groupby(glacier_hist_data.date.dt.dayofyear).mean()
whatif_data = whatif_data.rolling(date=15, center=True).mean()
whatif_data = whatif_data.groupby(whatif_data.date.dt.dayofyear).mean()

# Rename warming level coordinates
future_data_mean = future_data.groupby('warming_level').mean().assign_coords(warming_level=warming_levels)
future_data_sd   = future_data.groupby('warming_level').std().assign_coords(warming_level=warming_levels)
whatif_data_mean = whatif_data.groupby('warming_level').mean().assign_coords(warming_level=warming_levels)
whatif_data_sd   = whatif_data.groupby('warming_level').std().assign_coords(warming_level=warming_levels)

## Plot

In [51]:
colors = ['#2E86AB',  '#F18F01', "#C7231D"]  # Blue, Yellow, Red

# Create subplots with 1 row and 2 columns
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08)

# Plot observed data from glambie_data
fig.add_trace(go.Scatter(
    x=glambie_data.index,
    y=glambie_data['volume_norm'] * 100,
    mode='lines',
    name='Observed (GlaMBIE)',
    line=dict(color='grey', width=2, dash='dot')
), row=1, col=1)

# First subplot - Historical data
fig.add_trace(go.Scatter(
    x=hist_data.time, 
    y=hist_data.values * 100, 
    mode='lines',
    name='Historical (2000-2024)',
    line=dict(color='black', width=2)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=glacier_hist_data.dayofyear, 
    y=glacier_hist_data.values, 
    mode='lines',
    showlegend=False,
    line=dict(color='black', width=2)
), row=1, col=2)

# Plot future scenarios with confidence intervals for both subplots
for i, warming_level in enumerate(warming_levels):
    # First subplot data
    mean_data_vol = future_data_mean.sel(warming_level=warming_level)
    std_data_vol = future_data_sd.sel(warming_level=warming_level)
    
    # Second subplot data
    mean_data_melt = whatif_data_mean.sel(warming_level=warming_level)
    std_data_melt = whatif_data_sd.sel(warming_level=warming_level)
    
    # Convert hex to rgba for fillcolor
    hex_color = colors[i].lstrip('#')
    rgb = tuple(int(hex_color[j:j+2], 16) for j in (0, 2, 4))
    rgba_color = f'rgba({rgb[0]}, {rgb[1]}, {rgb[2]}, 0.2)'
    
    # First subplot - Volume confidence interval
    upper_vol = (mean_data_vol + std_data_vol) * 100
    lower_vol = (mean_data_vol - std_data_vol) * 100
    
    fig.add_trace(go.Scatter(
        x=list(mean_data_vol.time) + list(mean_data_vol.time)[::-1],
        y=list(upper_vol.values) + list(lower_vol.values)[::-1],
        fill='toself',
        fillcolor=rgba_color,
        line=dict(color='rgba(255,255,255,0)'),
        showlegend=False,
        hoverinfo="skip",
        name=f'+{warming_level} warming (±1σ)'
    ), row=1, col=1)
    
    # First subplot - Volume mean line
    fig.add_trace(go.Scatter(
        x=mean_data_vol.time, 
        y=mean_data_vol.values * 100,  # Convert to percentage
        mode='lines',
        name=f'+{warming_level} warming (N={ensemble_sizes[i]})',
        line=dict(color=colors[i], width=2)
    ), row=1, col=1)
    
    # Second subplot - Melt confidence interval
    upper_melt = mean_data_melt + std_data_melt
    lower_melt = mean_data_melt - std_data_melt
    
    fig.add_trace(go.Scatter(
        x=list(mean_data_melt.dayofyear) + list(mean_data_melt.dayofyear)[::-1],
        y=list(upper_melt.values) + list(lower_melt.values)[::-1],
        fill='toself',
        fillcolor=rgba_color,
        line=dict(color='rgba(255,255,255,0)'),
        showlegend=False,
        hoverinfo="skip",
        name=f'+{warming_level} warming (±1σ)'
    ), row=1, col=2)
    
    # Second subplot - Melt mean line
    fig.add_trace(go.Scatter(
        x=mean_data_melt.dayofyear, 
        y=mean_data_melt.values,
        mode='lines',
        name=f'+{warming_level} warming (N={ensemble_sizes[i]})',
        line=dict(color=colors[i], width=2),
        showlegend=False
    ), row=1, col=2)


# Update x and y axes
fig.update_xaxes(title_text='Year', griddash="dot", range=[2000, 2100], ticks="outside", row=1, col=1)
fig.update_xaxes(
    title_text='Day of year',
    griddash="dot",
    ticks="outside",
    tickvals=[15,46,74,105,135,166,196,227,258,288,319,349],
    ticktext=["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"],
    range=[1, 365],
    row=1, col=2
)

fig.update_yaxes(title_text='Glacier volume (%)', title_standoff=0, range=[0, 120], griddash="dot", ticks="outside", row=1, col=1)
fig.update_yaxes(title_text='Glacier melt (m<sup>3</sup>s<sup>-1</sup>)', title_standoff=0, range = [0, 15000], griddash="dot", ticks="outside", row=1, col=2)

# Add subplot annotations
fig.add_annotation(text="a)", xref="paper", yref="paper", x=0.44, y=1, showarrow=False, font=dict(size=18))
fig.add_annotation(text="b)", xref="paper", yref="paper", x=0.99, y=1, showarrow=False, font=dict(size=18))

# Update layout
fig.update_layout(
    width=900,
    height=500, font_family="Arial",
    font=dict(size=15),
    plot_bgcolor="rgba(229,236,246,0.7)",
    margin=dict(l=10, r=10, b=10, t=10),
    legend=dict(yanchor="top", y=0.31, x=0.01, bgcolor="rgba(255,255,255,0.7)"))

fig.write_image("figures/fig_glacier_evolution.svg", width=900, height=500)
fig.show()